# 🎤 Vocalido DiffSinger Training (A100)
**Upload → Set A100 Runtime → Run All**

In [ ]:
# Cell 0: GPU + Drive
AUTO_SHUTDOWN_WHEN_DONE = True
import torch, json, time, os
assert torch.cuda.is_available(), 'No GPU! Change Runtime to A100 first.'
gpu_name = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name}')
hourly_rate = 3.00 if 'A100' in gpu_name else 0.80 if 'L4' in gpu_name else 0.50
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
PROGRESS_FILE = '/content/drive/MyDrive/vocalido_progress.json'
SESSION_START = time.time()
def save_progress(phase, pct, detail, gpu_on=True):
    try:
        hrs = (time.time() - SESSION_START) / 3600
        with open(PROGRESS_FILE, 'w') as f:
            json.dump({'phase': phase, 'pct': pct, 'detail': detail,
                       'time': time.time(), 'gpu_active': gpu_on,
                       'est_cost_usd': hrs * hourly_rate if gpu_on else 0}, f)
    except: pass
save_progress('preparing', 5, f'{gpu_name} Connected', True)
print('Drive mounted OK.')

In [ ]:
# Cell 1: Install DiffSinger
save_progress('preparing', 10, 'Installing DiffSinger...', True)
!rm -rf /content/DiffSinger
!git clone https://github.com/openvpi/DiffSinger.git /content/DiffSinger -q
%cd /content/DiffSinger
!pip install -q -r requirements.txt
!pip install -q praat-parselmouth pyworld pyloudnorm tensorboard
print('DiffSinger installed.')

In [ ]:
# Cell 2: Install MFA (Colab-compatible — no conda needed)
save_progress('preparing', 20, 'Installing MFA...', True)
import os

# Install Miniforge (provides conda on Colab)
!wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
!bash /tmp/miniforge.sh -b -p /opt/conda > /dev/null 2>&1
os.environ['PATH'] = '/opt/conda/bin:' + os.environ.get('PATH', '')
!/opt/conda/bin/conda install -c conda-forge montreal-forced-aligner -y -q > /dev/null 2>&1
# Verify and link
import subprocess
check = subprocess.run(['/opt/conda/bin/mfa', 'version'], capture_output=True, text=True)
if check.returncode == 0:
    !ln -sf /opt/conda/bin/mfa /usr/local/bin/mfa
    print(f'✅ MFA ready: {check.stdout.strip()}')
else:
    print('⚠️ Fallback: pip MFA')
    !pip install -q praatio montreal-forced-aligner 2>/dev/null || true
save_progress('preparing', 25, 'MFA ready', True)
print('Cell 2 done.')

In [ ]:
# Cell 3: Dataset Setup
%cd /content/DiffSinger
save_progress('preparing', 30, 'Loading dataset...', True)
import shutil, glob, os
DS_DIR = '/content/DiffSinger/data/vocalido'
os.makedirs(f'{DS_DIR}/wavs', exist_ok=True)

# Search all possible dataset locations
wavs_found = []
for search_path in [
    '/content/drive/MyDrive/vocalido_dataset/wavs',
    '/content/drive/MyDrive/vocalido_dataset',
    '/content/drive/MyDrive/Vocalido Voice',
    '/content/drive/MyDrive/diffsinger_dataset/wavs',
    '/content/wavs',
    '/tmp/wavs',
]:
    if os.path.exists(search_path):
        found = glob.glob(f'{search_path}/**/*.wav', recursive=True)
        if found:
            wavs_found = found
            print(f'Found {len(found)} WAV files in: {search_path}')
            break

if not wavs_found:
    print('⚠️  No WAV files found in Google Drive!')
    print('Please upload your wavs.zip using the cell below, then re-run this cell.')
    from google.colab import files as colab_files
    import zipfile
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        with open(f'/tmp/{fname}', 'wb') as f:
            f.write(data)
        if fname.endswith('.zip'):
            with zipfile.ZipFile(f'/tmp/{fname}') as z:
                z.extractall(f'{DS_DIR}/wavs')
        elif fname.endswith('.wav'):
            shutil.copy(f'/tmp/{fname}', f'{DS_DIR}/wavs/{fname}')
    wavs_found = glob.glob(f'{DS_DIR}/wavs/**/*.wav', recursive=True)

# Copy to dataset dir
for wp in wavs_found:
    dest = f'{DS_DIR}/wavs/{os.path.basename(wp)}'
    if not os.path.exists(dest):
        shutil.copy(wp, dest)

n_wav = len([f for f in os.listdir(f'{DS_DIR}/wavs') if f.endswith('.wav')])
print(f'Total WAV in dataset: {n_wav}')
assert n_wav > 0, 'No WAV files! Please upload your dataset.'

# Copy transcriptions if available
for t in ['transcriptions.txt', 'transcriptions.json']:
    for src in [f'/content/drive/MyDrive/{t}', f'/content/drive/MyDrive/vocalido_dataset/{t}']:
        if os.path.exists(src):
            shutil.copy(src, f'{DS_DIR}/{t}')
            print(f'Copied {t}')
            break

# Create .lab files (required by MFA)
for wav in os.listdir(f'{DS_DIR}/wavs'):
    if wav.endswith('.wav'):
        lab = f'{DS_DIR}/wavs/{wav}'.replace('.wav', '.lab')
        if not os.path.exists(lab):
            open(lab, 'w').write('ah')

save_progress('preparing', 35, f'{n_wav} WAV files ready', True)
print(f'Dataset ready: {n_wav} files.')

In [ ]:
# Cell 4: MFA Alignment
save_progress('align', 0, 'Downloading MFA models...', True)
!mfa model download acoustic english_mfa
!mfa model download dictionary english_mfa
save_progress('align', 50, 'Aligning...', True)
!mfa align {DS_DIR}/wavs english_mfa english_mfa {DS_DIR}/textgrids --clean -j 4
print('Alignment done.')

In [ ]:
# Cell 5: Config + Vocoder
%cd /content/DiffSinger
save_progress('preprocess', 0, 'Config + Vocoder...', True)
import yaml, os
DS_DIR = '/content/DiffSinger/data/vocalido'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
batch_size = 48 if vram_gb > 30 else 16
print(f'VRAM: {vram_gb:.1f} GB  |  Batch: {batch_size}')

!mkdir -p /content/vocoder/nsf_hifigan
!wget -q 'https://github.com/openvpi/vocoders/releases/download/nsf-hifigan-44.1k-hop512-128bin-2024.02/nsf_hifigan_44.1k_hop512_128bin_2024.02.zip' -O /tmp/voc.zip
!unzip -o -q /tmp/voc.zip -d /content/vocoder/nsf_hifigan
print('Vocoder ready.')

config = {
    'base_config': 'configs/acoustic.yaml',
    'task_cls': 'training.acoustic_task.AcousticTask',
    'raw_data_dir': DS_DIR,
    'binary_data_dir': f'{DS_DIR}_bin',
    'dictionary': 'dictionaries/english-dsdict.txt',
    'vocoder': 'nsf_hifigan',
    'vocoder_ckpt': '/content/vocoder/nsf_hifigan',
    'speakers': ['vocalido'], 'spk_id': 0, 'num_spk': 1,
    'audio_sample_rate': 44100, 'hop_size': 512, 'win_size': 2048, 'fft_size': 2048,
    'audio_num_mel_bins': 128, 'f0_min': 65, 'f0_max': 1100,
    'max_batch_size': batch_size,
    'max_batch_frames': 160000 if vram_gb > 30 else 80000,
    'max_epochs': 2000, 'num_ckpt_keep': 5, 'val_check_interval': 500,
    'optimizer_args': {'lr': 0.0004},
    'diff_decoder_type': 'wavenet', 'K_step': 100, 'timesteps': 100,
    'use_spk_embed': False, 'use_spk_id': True
}
os.makedirs('usr/configs', exist_ok=True)
with open('usr/configs/vocalido.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
save_progress('preprocess', 10, 'Config saved', True)
print('Config saved.')

In [ ]:
# Cell 6: Preprocess
%cd /content/DiffSinger
save_progress('preprocess', 50, 'Binarizing...', True)
!python scripts/binarize.py --config usr/configs/vocalido.yaml
save_progress('preprocess', 100, 'Binarization done', True)
print('Binarization done.')

In [ ]:
# Cell 7: Train (Auto-Resume)
%cd /content/DiffSinger
save_progress('training', 0, f'Training on {gpu_name}...', True)
CKPT_PATH = '/content/DiffSinger/checkpoints/vocalido_v1'
# Also save checkpoints to Drive so they survive session restart
import os
DRIVE_CKPT = '/content/drive/MyDrive/vocalido_checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(CKPT_PATH, exist_ok=True)
# Restore from Drive if available
import glob, shutil
drive_ckpts = glob.glob(f'{DRIVE_CKPT}/*.ckpt')
if drive_ckpts and not os.listdir(CKPT_PATH):
    for c in drive_ckpts: shutil.copy(c, CKPT_PATH)
    print(f'Restored {len(drive_ckpts)} checkpoints from Drive')
if os.path.exists(CKPT_PATH) and len(os.listdir(CKPT_PATH)) > 0:
    print('Resuming from checkpoint...')
    !python scripts/train.py --config usr/configs/vocalido.yaml --exp_name vocalido_v1
else:
    print('Starting fresh training...')
    !python scripts/train.py --config usr/configs/vocalido.yaml --exp_name vocalido_v1 --reset
# Backup checkpoints to Drive after each run
for c in glob.glob(f'{CKPT_PATH}/*.ckpt'): shutil.copy(c, DRIVE_CKPT)
save_progress('training', 100, 'Training done!', True)
print('Training complete!')

In [ ]:
# Cell 8: Export ONNX
save_progress('exporting', 0, 'Exporting...', True)
ONNX_DIR = '/content/vocalido_onnx'
os.makedirs(ONNX_DIR, exist_ok=True)
!python scripts/export.py --exp_name vocalido_v1 --out {ONNX_DIR}
import glob
for voc in glob.glob('/content/vocoder/**/*.onnx', recursive=True): shutil.copy(voc, ONNX_DIR)
DRIVE_OUT = '/content/drive/MyDrive/vocalido_models'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in os.listdir(ONNX_DIR): shutil.copy(f'{ONNX_DIR}/{f}', DRIVE_OUT)
save_progress('done', 100, 'All done! Models in Google Drive.', False)
print(f'Done! {len(os.listdir(ONNX_DIR))} models saved.')

In [ ]:
# Cell 9: Auto-Shutdown
if AUTO_SHUTDOWN_WHEN_DONE:
    print('Shutting down GPU to save costs...')
    save_progress('done', 100, 'GPU Off - Training Complete', False)
    import time; time.sleep(3)
    from google.colab import runtime
    runtime.unassign()
else:
    print('Done! Remember to disconnect GPU manually.')